# DeepFusionGuard — Colab setup

Run this at the start of **every** Colab session (Runtime → Run all). It is safe to re-run.

**Before the first run:** Runtime → Change runtime type → **T4 GPU**.

What it does:
1. Checks the GPU
2. Mounts Google Drive and creates the project folders there
3. Clones (or updates) the repo
4. Installs our exact pinned packages (torch 2.11, same as the laptop) with the right CUDA build for this GPU
5. Prints the environment report
6. Defines `stage_in` / `stage_out`: copy archives Drive ⇄ fast local disk

**Rules**
- Code is edited on the laptop in VS Code → `git push` → this notebook pulls it. Do not edit code here.
- Run project code as `!python scripts/...` (fresh process). If you import numpy/torch *inside* the notebook, do Runtime → Restart session once after the install cell.
- Never read thousands of small files straight from `/content/drive`; use `stage_in` first.
- Kaggle backup: prefer **T4 x2**. A P100 also works: the install cell detects it and installs torch's CUDA 12.6 build, the only build that still supports P100.
- First run of a fresh session downloads the CUDA build of torch (roughly 2 GB, over Google's network, not yours); it takes a few minutes.


In [ ]:
# ==== Settings: the only place to edit paths for Colab ====
# configs/default.yaml (profile 'colab') reads DRIVE_ROOT and LOCAL_ROOT from the env vars set below.
REPO_URL   = "https://github.com/kritikapandey1/Major_project.git"
BRANCH     = "main"
REPO_DIR   = "/content/Major_project"                 # code, re-cloned each session
DRIVE_ROOT = "/content/drive/MyDrive/DeepFusionGuard"  # persistent storage
LOCAL_ROOT = "/content/dfg_work"                      # fast local disk, wiped each session
DRIVE_SUBDIRS = ["archives/raw", "archives/processed", "features", "checkpoints", "logs", "hf_cache"]


## 1. GPU check

In [ ]:
import shutil, subprocess

if shutil.which("nvidia-smi"):
    print(subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total,driver_version",
                          "--format=csv,noheader"], capture_output=True, text=True).stdout)
else:
    print("No GPU attached. Runtime -> Change runtime type -> T4 GPU, then Run all again.\n"
          "(Setup still works on CPU; only heavy steps need the GPU.)")


## 2. Mount Drive + create project folders

In [ ]:
import os
from google.colab import drive

drive.mount("/content/drive")
for sub in DRIVE_SUBDIRS:
    os.makedirs(os.path.join(DRIVE_ROOT, sub), exist_ok=True)
os.makedirs(LOCAL_ROOT, exist_ok=True)

# Inherited by every `!python ...` command below.
os.environ["HF_HOME"] = os.path.join(DRIVE_ROOT, "hf_cache")  # models download once, reused every session
os.environ["DFG_DRIVE_ROOT"] = DRIVE_ROOT
os.environ["DFG_LOCAL_ROOT"] = LOCAL_ROOT
os.environ["DFG_PROFILE"] = "colab"  # configs/default.yaml -> profiles.colab
print("Drive folders ready:", sorted(os.listdir(DRIVE_ROOT)))


## 3. Clone or update the repo

In [ ]:
import os, subprocess

if os.path.isdir(os.path.join(REPO_DIR, ".git")):
    subprocess.run(["git", "-C", REPO_DIR, "pull", "--ff-only"], check=True)
else:
    subprocess.run(["git", "clone", "--branch", BRANCH, REPO_URL, REPO_DIR], check=True)
os.chdir(REPO_DIR)
print(subprocess.run(["git", "log", "--oneline", "-3"], capture_output=True, text=True).stdout)

missing = [f for f in ["requirements.txt", "scripts/env_report.py"] if not os.path.exists(f)]
if missing:
    raise FileNotFoundError(f"{missing} not in the repo yet: commit and push the Step 0 files from your laptop first.")


## 4. Install packages

We install **our exact pins** from `requirements.txt`, including torch 2.11.0, instead of Colab's preinstalled torch. Colab's runtime changes every few months (newer torch, and torchaudio is being removed), so pinning keeps laptop, Colab and Kaggle identical and results reproducible.

The CUDA build is chosen from the GPU's compute capability: P100 and older (below 7.5) need the CUDA 12.6 build; T4 and newer use `auto` (uv picks the newest build the driver supports); no GPU means CPU build. All OpenCV variants are removed first and only `opencv-python-headless` is reinstalled, because they share one `cv2` folder.

In [ ]:
%%bash
set -euo pipefail
pip install -q uv

CC=$(nvidia-smi --query-gpu=compute_cap --format=csv,noheader 2>/dev/null | head -n1 | tr -d ' ' || true)
if [ -z "$CC" ]; then
    BACKEND=cpu
elif awk -v c="$CC" 'BEGIN { exit !(c < 7.5) }'; then
    BACKEND=cu126   # Pascal/Volta (e.g. Kaggle P100): only CUDA 12.6 builds support them
else
    BACKEND=auto    # Turing+ (T4, L4, A100): newest build the driver supports
fi
echo "GPU compute capability: ${CC:-none} -> torch backend: $BACKEND"

pip uninstall -y -q opencv-python opencv-contrib-python opencv-python-headless || true
uv pip install --system --break-system-packages -q -r requirements.txt --torch-backend "$BACKEND"
echo "Install finished."
python -c "import torch, torchaudio, numpy; print('torch', torch.__version__, '| torchaudio', torchaudio.__version__, '| numpy', numpy.__version__)"

## 5. Environment report

In [ ]:
!python scripts/env_report.py --device auto


In [ ]:
# Resolved config for this session (profile, device, paths on local disk + Drive, models)
!python scripts/show_config.py

## 6. Data staging helpers (Drive ⇄ fast local disk)

Data lives on Drive as a few big archives (`.tar` / `.zip`). At the start of a session, `stage_in` copies one to `/content` and extracts it; `stage_out` packs a local result folder and saves it back to Drive.

In [ ]:
import shutil, time
from pathlib import Path


def stage_in(archive_name: str, subdir: str = "archives/raw") -> Path:
    """Copy DRIVE_ROOT/subdir/archive_name to local disk and extract it.

    Returns the local folder. Skips the work if it was already staged this session.
    """
    src = Path(DRIVE_ROOT, subdir, archive_name)
    if not src.exists():
        raise FileNotFoundError(f"{src} not found on Drive")
    stem = src.name
    for ext in (".tar.gz", ".tgz", ".tar", ".zip"):
        stem = stem.removesuffix(ext)
    dest = Path(LOCAL_ROOT, stem)
    marker = dest / ".staged"
    if marker.exists():
        print(f"already staged: {dest}")
        return dest
    t0 = time.time()
    local_copy = Path(LOCAL_ROOT, src.name)
    shutil.copy2(src, local_copy)
    dest.mkdir(parents=True, exist_ok=True)
    if local_copy.suffix == ".zip":
        shutil.unpack_archive(local_copy, dest)
    else:  # tar family: 'data' filter blocks unsafe paths inside the archive
        shutil.unpack_archive(local_copy, dest, filter="data")
    local_copy.unlink()
    marker.touch()
    print(f"staged {src.name} -> {dest} in {time.time() - t0:.1f} s")
    return dest


def stage_out(local_dir: str | Path, archive_name: str, subdir: str = "archives/processed") -> Path:
    """Pack local_dir into an uncompressed .tar and save it to DRIVE_ROOT/subdir.

    Writes to a .part file first, then renames, so an interrupted session never
    leaves a half-written archive that looks complete.
    """
    local_dir = Path(local_dir)
    if not local_dir.is_dir():
        raise FileNotFoundError(f"{local_dir} is not a folder")
    if not archive_name.endswith(".tar"):
        raise ValueError("archive_name must end with .tar")
    t0 = time.time()
    tmp = Path(shutil.make_archive(str(Path(LOCAL_ROOT, archive_name[:-4])), "tar", root_dir=local_dir))
    dst = Path(DRIVE_ROOT, subdir, archive_name)
    part = dst.with_name(dst.name + ".part")
    shutil.copy2(tmp, part)
    part.replace(dst)
    tmp.unlink()
    print(f"saved {local_dir} -> {dst} ({dst.stat().st_size / 2**20:.1f} MB) in {time.time() - t0:.1f} s")
    return dst


In [ ]:
# Round-trip smoke test: local folder -> Drive archive -> back to local. Cleans up after itself.
import filecmp

demo = Path(LOCAL_ROOT, "_smoke_src")
demo.mkdir(parents=True, exist_ok=True)
(demo / "hello.txt").write_text("DeepFusionGuard staging test\n")

stage_out(demo, "_smoke.tar", subdir="logs")
back = stage_in("_smoke.tar", subdir="logs")
assert filecmp.cmp(demo / "hello.txt", back / "hello.txt", shallow=False), "round-trip mismatch"
print("Staging round-trip OK")

shutil.rmtree(demo); shutil.rmtree(back); Path(DRIVE_ROOT, "logs", "_smoke.tar").unlink()


## Next steps (cells added as we complete each step)
- Step 2: preprocessing (`!python scripts/preprocess.py ...`)
- Step 3: feature extraction + caching to `DRIVE_ROOT/features`
- Step 5: training, checkpoints to `DRIVE_ROOT/checkpoints`
